In [1]:
import os
import io
import shutil
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras.preprocessing import image

print(f"TensorFlow version: {tf.__version__}")

I0000 00:00:1786637444.736039     317 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1786637444.746647     317 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1786637445.554640     317 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1786637448.386916     317 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

TensorFlow version: 2.21.0


In [2]:
TRAIN_DIR = "../data/fruits_data/train"
VAL_DIR = "../data/fruits_data/validation"
BATCH_SIZE = 8
IMG_SIZE = (224, 224)  # VGG16 strictly demands 224x224 input arrays
EPOCHS = 42

print("\n--- Loading Datasets ---")
train_ds = image_dataset_from_directory(
    TRAIN_DIR,
    shuffle=True,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    label_mode='categorical'
)

val_ds = image_dataset_from_directory(
    VAL_DIR,
    shuffle=False,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    label_mode='categorical'
)

class_names = train_ds.class_names
num_classes = len(class_names)
print(f"Target Labels parsed from directory structure: {class_names}")



--- Loading Datasets ---
Found 3115 files belonging to 36 classes.
Found 351 files belonging to 36 classes.
Target Labels parsed from directory structure: ['apple', 'banana', 'beetroot', 'bell pepper', 'cabbage', 'capsicum', 'carrot', 'cauliflower', 'chilli pepper', 'corn', 'cucumber', 'eggplant', 'garlic', 'ginger', 'grapes', 'jalepeno', 'kiwi', 'lemon', 'lettuce', 'mango', 'onion', 'orange', 'paprika', 'pear', 'peas', 'pineapple', 'pomegranate', 'potato', 'raddish', 'soy beans', 'spinach', 'sweetcorn', 'sweetpotato', 'tomato', 'turnip', 'watermelon']


E0000 00:00:1786637451.769039     317 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [3]:
print("\n--- Building VGG16 Architecture ---")
# 1. Download base weights without default 1000-class head
vgg16_base = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# 2. Freeze base layers to keep general features (edges, color boundaries) locked
vgg16_base.trainable = False

# 3. Build sequential container adding custom classification head
model = models.Sequential([
    # VGG16 expects image data formatted explicitly by its own library functions
    layers.Lambda(preprocess_input, input_shape=(224, 224, 3)),
    vgg16_base,
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.2),  # Mitigates overfitting behavior
    layers.Dense(num_classes, activation='softmax')  # Yields prediction probabilities
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1), # Sharpens class boundaries
    metrics=['accuracy']
)

model.summary()


--- Building VGG16 Architecture ---
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


/usr/local/lib/python3.11/site-packages/keras/src/layers/core/lambda_layer.py:65: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lambda (Lambda)                 │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vgg16 (Functional)              │ (None, 7, 7, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     6,422,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 36)             │         9,252 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,146,724 (80.67 MB)

 Trainable params: 6,432,036 (24.54 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [ ]:
print("\n--- Initiating Model Fit Loop ---")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

# Render accuracy and loss analytics side-by-side
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(len(acc))

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy', marker='o')
plt.plot(epochs_range, val_acc, label='Validation Accuracy', marker='x')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss', marker='o')
plt.plot(epochs_range, val_loss, label='Validation Loss', marker='x')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.grid(True)
plt.show()

# Save final weights to disk
model.save("../ML_models/fruit_veg_vgg16_model.keras")
print("Saved baseline model artifact to disk as 'fruit_veg_vgg16_model.keras'")


--- Initiating Model Fit Loop ---
Epoch 1/42


/usr/local/lib/python3.11/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
W0000 00:00:1786637457.091570     396 png_io.cc:96] PNG warning: iCCP: known incorrect sRGB profile


  6/390 ━━━━━━━━━━━━━━━━━━━━ 5:09 807ms/step - accuracy: 0.0417 - loss: 46.1391

W0000 00:00:1786637462.530084     395 png_io.cc:96] PNG warning: iCCP: known incorrect sRGB profile


  8/390 ━━━━━━━━━━━━━━━━━━━━ 5:07 804ms/step - accuracy: 0.1250 - loss: 47.9855

...

274/390 ━━━━━━━━━━━━━━━━━━━━ 1:57 1s/step - accuracy: 0.2997 - loss: 2.8606

W0000 00:00:1786646435.899738    2834 png_io.cc:96] PNG warning: iCCP: known incorrect sRGB profile


281/390 ━━━━━━━━━━━━━━━━━━━━ 1:50 1s/step - accuracy: 0.3003 - loss: 2.8724

W0000 00:00:1786646443.016042    2839 png_io.cc:96] PNG warning: iCCP: known incorrect sRGB profile


319/390 ━━━━━━━━━━━━━━━━━━━━ 1:12 1s/step - accuracy: 0.2955 - loss: 2.8757

In [ ]:
def predict_single_image(img_path: str):
    """Utility function to load, print, and identify a single photo."""
    print(f"\n--- Evaluation Request for: {img_path} ---")
    if not os.path.exists(img_path):
        print(f"Error: Target path {img_path} does not exist.")
        return
        
    # Read image payload from file path
    img = image.load_img(img_path, target_size=IMG_SIZE)
    
    # In-line graph display inside Jupyter window
    plt.imshow(img)
    plt.axis('off')
    plt.show()
    
    # Transform into numerical matrix and construct batch dimension [1, 224, 224, 3]
    img_array = image.img_to_array(img)
    img_batch = np.expand_dims(img_array, axis=0)
    
    # Execute matrix mathematics
    predictions = model.predict(img_batch)[0]
    best_match_idx = np.argmax(predictions)
    confidence_score = predictions[best_match_idx]
    
    print(f"RESULT: Identified as '{class_names[best_match_idx]}' with {100 * confidence_score:.2f}% confidence.")

    for idx,class_name in enumerate(class_names):
        print(f"{class_name}: {predictions[idx] * 100:.2f}%")

# Test the prediction logic immediately using an image from our validation set
sample_test_path = "./hello.jpg"
predict_single_image(sample_test_path)

In [ ]:
def predict_image(img_path: str):
    """Utility function to load and identify a single photo."""

    print(f"\n--- Evaluation Request for: {img_path} ---")

    if not os.path.exists(img_path):
        print(f"Error: Target path {img_path} does not exist.")
        return

    # Read image
    img = image.load_img(
        img_path,
        target_size=IMG_SIZE
    )

    # Display image
    plt.imshow(img)
    plt.axis("off")
    plt.show()

    # Convert image to numerical matrix
    img_array = image.img_to_array(img)

    # Add batch dimension [1, 224, 224, 3]
    img_batch = np.expand_dims(img_array, axis=0)

    # Execute prediction
    predictions = model.predict(img_batch)[0]

    # Get indices of top 3 predictions
    top_three_indices = np.argsort(predictions)[-3:][::-1]
        # Build prediction result
    prediction_result = {
        "fileName": file_name,
        "predictions": []
    }

    print("\nTop 3 Predictions:")

    for rank, idx in enumerate(top_three_indices, start=1):

        class_name = class_names[idx]
        confidence_score = predictions[idx]

        print(
            f"{rank}. {class_name}: "
            f"{confidence_score * 100:.2f}%"
        )
        prediction_result["predictions"].append({
            "rank":rank,
            "class":class_name[idx],
            "confidence":round(fload(predictions[idx]*100),
                               2)
        })

In [ ]:
import os
import json


trial_images = "../trialImaages"
data_folder = "../data"

output_file = os.path.join(
    data_folder,
    "fruit_vegetable_predictions.json"
)

image_extensions = (
    ".jpg",
    ".jpeg",
    ".png",
    ".webp"
)


def generate_predictions():

    os.makedirs(
        data_folder,
        exist_ok=True
    )

    results = []

    for file_name in os.listdir(trial_images):

        if not file_name.lower().endswith(
            image_extensions
        ):
            continue

        image_path = os.path.join(
            trial_images,
            file_name
        )

        prediction = predict_image(
            image_path
        )

        if prediction is not None:
            results.append(prediction)

    with open(
        output_file,
        "w",
        encoding="utf-8"
    ) as json_file:

        json.dump(
            results,
            json_file,
            indent=4
        )

    print(
        f"Predictions saved to: {output_file}"
    )



In [ ]:
generate_predictions()